# DX 704 Week 9 Project

This week's project will build an email spam classifier based on the Enron email data set.
You will perform your own feature extraction, and use naive Bayes to estimate the probability that a particular email is spam or not.
Finally, you will review the tradeoffs from different thresholds for automatically sending emails to the junk folder.

The full project description and a template notebook are available on GitHub: [Project 9 Materials](https://github.com/bu-cds-dx704/dx704-project-09).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download Data Set

We will be using the Enron spam data set as prepared in this GitHub repository.

https://github.com/MWiechmann/enron_spam_data

You may need to download this differently depending on your environment.

In [1]:
!wget https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip

--2026-11-03 22:25:13--  https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/MWiechmann/enron_spam_data/refs/heads/master/enron_spam_data.zip [following]
--2026-11-03 22:25:13--  https://raw.githubusercontent.com/MWiechmann/enron_spam_data/refs/heads/master/enron_spam_data.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15642124 (15M) [application/zip]
Saving to: ‘enron_spam_data.zip.1’

enron_spam_data.zip 100%[===================>]  14.92M  51.1MB/s    in 0.3s    

2026-11-03 22:25:14 (51.1 MB/s) - ‘enron_s

In [10]:
%pip install Pandas 
%pip install numpy

import pandas as pd
import numpy as np

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
# pandas can read the zip file directly
enron_spam_data = pd.read_csv("enron_spam_data.zip")
enron_spam_data

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
...,...,...,...,...,...
33711,33711,= ? iso - 8859 - 1 ? q ? good _ news _ c = eda...,"hello , welcome to gigapharm onlinne shop .\np...",spam,2005-07-29
33712,33712,all prescript medicines are on special . to be...,i got it earlier than expected and it was wrap...,spam,2005-07-29
33713,33713,the next generation online pharmacy .,are you ready to rock on ? let the man in you ...,spam,2005-07-30
33714,33714,bloow in 5 - 10 times the time,learn how to last 5 - 10 times longer in\nbed ...,spam,2005-07-30


In [ ]:
(enron_spam_data["Spam/Ham"] == "spam").mean()

np.float64(0.5092834262664611)

## Part 2: Design a Feature Extractor

Design a feature extractor for this data set and write out two files of features based on the text.
Don't forget that both the Subject and Message columns are relevant sources of text data.
For each email, you should count the number of repetitions of each feature present.
The auto-grader will assume that you are using a multinomial distribution in the following problems.

In [ ]:
# ===================================
# Part 2: Design a Feature Extractor
# ===================================

import re
import json
from collections import Counter

TOKEN_RE = re.compile(r"[a-z][a-z0-9']*[a-z0-9]|[a-z]|[$!]")

def tokenize(text):
    if not isinstance(text, str):
        return []
    return TOKEN_RE.findall(text.lower())

def extract_features(subject, message):
    # Bag of words
    counts = Counter()
    for token in tokenize(subject):
        counts["subj_" + token] += 1
    for token in tokenize(message):
        counts["body_" + token] += 1
    return counts

Assign a row to the test data set if `Message ID % 30 == 0` and assign it to the training data set otherwise.
Write two files, "train-features.tsv" and "test-features.tsv" with two columns, Message ID and features_json.
The features_json column should contain a JSON dictionary where the keys are your feature names and the values are integer feature values.
This will give us a sparse feature representation.


In [ ]:
# =================================================
# Assign train and test sets based on Message ID
# =================================================

is_test = (enron_spam_data["Message ID"] % 30 == 0)
train_df = enron_spam_data[~is_test]
test_df = enron_spam_data[is_test]

def all_features(df):
    return [extract_features(s, m) for s, m in zip(df["Subject"], df["Message"])]

train_features = all_features(train_df)
test_features = all_features(test_df)

# Vocab from training data: keep features appearing in >= 5 training emails
doc_frequency = Counter()

for f in train_features:
    doc_frequency.update(f.keys())
vocab = {k for k, v in doc_frequency.items() if v >= 5}

def write_features(df, features, path):
    rows = []
    for mid, f in zip(df["Message ID"], features):
        d = {k: int(v) for k, v in f.items() if k in vocab}
        rows.append((mid, json.dumps(d, sort_keys=True)))
    with open(path, "w") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")


# =====================================================================
# Save to TSV files named 'train-features.tsv' and 'test-features.tsv'
# =====================================================================

write_features(train_df, train_features, "train-features.tsv")
write_features(test_df, test_features, "test-features.tsv")

Submit "train-features.tsv" and "test-features.tsv" in Gradescope.

Hint: these features will be graded based on the test accuracy of a logistic regression based on the training features.
This is to make sure that your feature set is not degenerate; you do not need to compute this regression yourself.
You can separately assess your feature quality based on your results in part 6.

## Part 3: Compute Conditional Probabilities

Based on your training data, compute appropriate conditional probabilities for use with naïve Bayes.
Use of additive smoothing with $\alpha=1$ to avoid zeros.


In [8]:
# ==========================================
# Part 3: Compute Conditional Probabilities
# ==========================================

# Based on training data, compute conditional probabilities for use with naive Bayes
# Use additive smoothing with alpha=1

def load_features(path):
    rows = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith("["):
                mid, fj = json.loads(line)
            else:
                mid, fj = line.split("\t", 1)
                if mid == "Message ID":
                    continue
                fj = fj[1:-1].replace('""', '"') if fj.startswith('"') else fj
            rows[int(mid)] = json.loads(fj)
    return rows

train_features = load_features("train-features.tsv")
label_of = enron_spam_data.set_index("Message ID")["Spam/Ham"]

counts = {"ham": Counter(), "spam": Counter()}
for mid, feats in train_features.items():
    counts[label_of[mid]].update(feats)

features = sorted(set(counts["ham"]) | set(counts["spam"]))
V = len(features)
alpha = 1
totals = {c: sum(counts[c].values()) for c in counts}

Save the conditional probabilities in a file "feature-probabilities.tsv" with columns feature, ham_probability and spam_probability.

In [9]:
# =============================================================================
# Save conditional probabilities to TSV file named 'feature-probabilities.tsv'
# =============================================================================

feature_probailities_df = pd.DataFrame({
    'feature': features,
    "ham_probability": [(counts["ham"][f] + alpha) / (totals["ham"] + alpha * V) for f in features],
    "spam_probability": [(counts["spam"][f] + alpha) / (totals["spam"] + alpha * V) for f in features],
})

# print(feature_probailities_df)

# Save file
feature_probailities_df.to_csv('feature-probabilities.tsv', sep='\t', index=False)
feature_probailities_df

,feature,ham_probability,spam_probability
0,body_!,2.344918e-03,8.723050e-03
1,body_$,4.469318e-03,6.789949e-03
2,body_a,1.530601e-02,1.555684e-02
3,body_aa,5.118744e-05,5.659068e-05
4,body_aaa,4.124106e-06,5.285943e-06
...,...,...,...
44037,subj_zisman,1.698161e-06,3.109378e-07
44038,subj_zol,2.425945e-07,3.420316e-06
44039,subj_zoloft,2.425945e-07,2.487503e-06
44040,subj_zone,2.183350e-06,4.042192e-06


Submit "feature-probabilities.tsv" in Gradescope.

## Part 4: Implement a Naïve Bayes Classifier

Implement a naïve Bayes classifier based on your previous feature probabilities.

In [26]:
# ==========================================
# Part 4: Implement Naive Bayes Classifier
# ==========================================

# Implement Naive Bayes classifier based on feature probabilities 
probs = pd.read_csv("feature-probabilities.tsv", sep="\t").set_index("feature")
log_ham = np.log(probs["ham_probability"]).to_dict()
log_spam = np.log(probs["spam_probability"]).to_dict()

train_labels = [label_of[m] for m in train_features]
prior_spam = np.mean([l == "spam" for l in train_labels])
log_prior = {"ham": np.log(1 - prior_spam), "spam": np.log(prior_spam)}

def predict_proba(features):
    # Multinomial Naive Bayes
    lh, ls = log_prior["ham"], log_prior["spam"]
    for f, c in features.items():
        if f in log_ham:
            lh += c * log_ham[f]
            ls += c * log_spam[f]
    m = max(lh, ls)
    eh, es = np.exp(lh - m), np.exp(ls - m)
    return eh / (eh + es), es / (eh + es)

def predict_all(features_by_id):
    rows = [(mid, *predict_proba(f)) for mid, f in features_by_id.items()]
    return pd.DataFrame(rows, columns=["Message ID", "ham", "spam"])

Save your prediction probabilities to "train-predictions.tsv" with columns Message ID, ham and spam.

In [ ]:
# ===========================================================================
# Save prediction probabilities to TSV file named 'train-predictions.tsv'
# ===========================================================================

train_features = load_features("train-features.tsv")
train_predictions = predict_all(train_features)
train_predictions_df = train_predictions

# Save file
train_predictions_df.to_csv('train-predictions.tsv', sep='\t', index=False)
train_predictions_df

,Message ID,ham,spam
0,1,1.000000e+00,8.639138e-167
1,2,1.000000e+00,9.637210e-13
2,3,1.000000e+00,2.836073e-148
3,4,1.000000e+00,3.996600e-142
4,5,1.000000e+00,1.641872e-37
...,...,...,...
32587,33711,1.197070e-18,1.000000e+00
32588,33712,1.218382e-36,1.000000e+00
32589,33713,2.457714e-17,1.000000e+00
32590,33714,2.027986e-04,9.997972e-01


Submit "train-predictions.tsv" in Gradescope.

## Part 5: Predict Spam Probability for Test Data

Use your previous classifier to predict spam probability for the test data.
If the test data has tokens that you did not model, just ignore them.

In [28]:
# ================================================
# Part 5: Predict Spam Probability for Test Data
# ================================================

# Use previous classifier to predict spam probabilities for test data
# If test data has tokens that were not seen during training, then ignore

test_features = load_features("test-features.tsv")
test_predictions = predict_all(test_features)

Save your prediction probabilities in "test-predictions.tsv" with the same columns as "train-predictions.tsv".

In [29]:
# ==========================================================================
# Save prediction probabilities to TSV file named 'test-predictions.tsv'
# ==========================================================================

test_predictions_df = test_predictions

# Save file
test_predictions_df.to_csv('test-predictions.tsv', sep='\t', index=False)
test_predictions_df

,Message ID,ham,spam
0,0,2.145899e-01,7.854101e-01
1,30,1.000000e+00,1.930975e-78
2,60,1.000000e+00,1.141388e-10
3,90,1.000000e+00,7.933419e-31
4,120,1.000000e+00,2.442242e-168
...,...,...,...
1119,33570,3.453386e-15,1.000000e+00
1120,33600,1.887142e-257,1.000000e+00
1121,33630,3.447076e-49,1.000000e+00
1122,33660,3.955566e-73,1.000000e+00


Submit "test-predictions.tsv" in Gradescope.

## Part 6: Construct ROC Curve

For every probability threshold from 0.01 to .99 in increments of 0.01, compute the false and true positive rates from the test data using the spam class for positives.
That is, if the predicted spam probability is greater than or equal to the threshold, predict spam.

In [34]:
# ============================
# Part 6: Construct ROC Curve
# ============================

test_predictions = pd.read_csv("test-predictions.tsv", sep="\t")
test_predictions["label"] = test_predictions["Message ID"].map(enron_spam_data.set_index("Message ID")["Spam/Ham"])

positives = test_predictions["label"] == "spam"
negatives = test_predictions["label"] != "spam"

roc_data = []

for i in range(1, 100):
    threshold = i / 100
    prediction = test_predictions["spam"] >= threshold

    true_positives = prediction & positives
    false_positives = prediction & negatives

    false_positive_rate = sum(false_positives) / sum(negatives)
    true_positive_rate = sum(true_positives) / sum(positives)

    roc_data.append((threshold, false_positive_rate, true_positive_rate))

Save this data in a file "roc.tsv" with columns threshold, false_positive_rate and true_positive rate.

In [35]:
# ==========================================================================
# Save data to TSV file named 'roc.tsv'
# ==========================================================================

roc_df = pd.DataFrame(roc_data, columns=["threshold", "false_positive_rate", "true_positive_rate"])

# Save file
roc_df.to_csv('roc.tsv', sep='\t', index=False)
roc_df

,threshold,false_positive_rate,true_positive_rate
0,0.01,0.043478,0.996503
1,0.02,0.043478,0.996503
2,0.03,0.041667,0.996503
3,0.04,0.039855,0.996503
4,0.05,0.039855,0.996503
...,...,...,...
94,0.95,0.028986,0.977273
95,0.96,0.028986,0.975524
96,0.97,0.028986,0.975524
97,0.98,0.027174,0.975524


Submit "roc.tsv" in Gradescope.

## Part 7: OpenAI API Key

Sign up for an OpenAI API acount at https://platform.openai.com/.
If you already have an OpenAI account (e.g. for ChatGPT), you may reuse that account.

Create a "dx704" project at https://platform.openai.com/settings/organization/projects, setup a $10/month spend limit, and create an API key for the project at https://platform.openai.com/api-keys.
You will use this API key for projects in later weeks.


No submission needed.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.